# The Tail Latency Cost of a Retransmission

Companion notebook for the [Packet Loss, Retransmission and Tail Latency](https://codemia.io/courses/networking_for_backend_engineers/net_packet_loss_retransmission_and_tail_latency) lesson on Codemia.

**What you will build**: the probability that a request touches a lost packet, the two very
different repairs that follow, and the percentile at which each one becomes visible. The headline
result is that a loss rate small enough to ignore per packet is not small per request, and that
where the loss falls in a transfer matters more than how much of it there is.

## Setup

In [ ]:
%pip install -q numpy matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(2024)

RTT_MS = 40
MIN_RTO_MS = 200        # the floor most stacks enforce on the retransmission timer
MSS = 1460

## Concept recap

A lost packet is repaired in one of two ways. If packets keep arriving behind the lost one, the
receiver acknowledges them out of order, the sender sees the duplicates and resends after roughly
one round trip. This is fast retransmit and it is cheap.

If nothing arrives behind the lost packet, because it was the last one of the response or because
the window was small, there are no duplicates to trigger anything and the only remaining mechanism
is the timer. That timer has a floor, typically 200ms, which on a 40ms path is five round trips of
nothing happening.

So the cost of a loss is not one number. It depends on whether there was traffic behind it.

## How likely is a request to be affected at all

In [ ]:
def request_loss_probability(packet_loss, packets):
    return 1 - (1 - packet_loss) ** packets

sizes = [(1, 'empty response'), (4, '5 KB'), (14, '20 KB'), (70, '100 KB'), (700, '1 MB')]

print(f"{'packet loss':>12}" + ''.join(f'{label:>16}' for _, label in sizes))
for loss in [0.0001, 0.001, 0.005, 0.01]:
    row = ''.join(f'{request_loss_probability(loss, n):>15.2%} ' for n, _ in sizes)
    print(f'{loss:>12.2%}' + row)

Read the 0.1% row. One packet in a thousand is the kind of loss rate that gets waved away, and a
100 KB response touches it 7% of the time. A one-megabyte response touches it half the time. The
conversion from per-packet to per-request is the step that is almost always skipped, and it is just
a power.

## The two repairs, and where they land

In [ ]:
def simulate_requests(n=300_000, packets=14, packet_loss=0.001, service_ms=12,
                      tail_loss_share=None, rng=rng):
    """Each request: service time, transfer time, and a repair if it loses a packet.

    A loss in the middle of a transfer is repaired by fast retransmit in about one round trip.
    A loss among the last few packets has nothing behind it and waits for the timer.
    """
    # with `packets` segments, the chance the lost one is near the end is roughly 3/packets
    tail_share = tail_loss_share if tail_loss_share is not None else min(1.0, 3.0 / packets)

    base = rng.lognormal(mean=np.log(service_ms), sigma=0.3, size=n) + RTT_MS
    affected = rng.random(n) < request_loss_probability(packet_loss, packets)
    at_tail = affected & (rng.random(n) < tail_share)
    fast = affected & ~at_tail

    latency = base.copy()
    latency[fast] += RTT_MS                                  # one extra round trip
    latency[at_tail] += MIN_RTO_MS + RTT_MS                  # wait for the timer, then resend
    return latency, fast, at_tail

latency, fast, at_tail = simulate_requests()
print(f'{fast.mean():.2%} of requests repaired by fast retransmit')
print(f'{at_tail.mean():.3%} of requests waited for the timer')
print()
for q in [50, 90, 95, 99, 99.9, 99.99]:
    print(f'p{q:<6g}{np.percentile(latency, q):>8.1f} ms')

Compare the percentiles against each other rather than against a target. The fast repairs are a
little over 1% of requests and they show up exactly where you would expect: the 99th percentile
sits one round trip above the 95th, which is a bounded, unremarkable cost. The timer waits are a
third of a percent of requests and they own the 99.9th percentile outright, three times the 99th.

A repair mechanism that fires rarely and costs 200ms every time it does is the classic shape of a
tail problem, and the ratio between p99 and p99.9 is its signature. A tail investigation that
starts from averages finds none of this.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].hist(latency[latency < 320], bins=140)
axes[0].axvline(np.percentile(latency, 99), color='tab:orange', linestyle='--', label='p99')
axes[0].axvline(np.percentile(latency, 99.9), color='tab:red', linestyle='--', label='p99.9')
axes[0].set_xlabel('latency (ms)')
axes[0].set_ylabel('requests')
axes[0].set_title('One bulk, one shoulder 200ms to the right')
axes[0].legend()

losses = np.logspace(-4, -1.3, 24)
for packets, label in [(14, '20 KB response'), (70, '100 KB'), (700, '1 MB')]:
    p99 = []
    p999 = []
    for loss in losses:
        sample, _, _ = simulate_requests(n=120_000, packets=packets, packet_loss=loss)
        p99.append(np.percentile(sample, 99))
        p999.append(np.percentile(sample, 99.9))
    axes[1].plot(losses, p99, label=f'{label} p99')
    axes[1].plot(losses, p999, linestyle='--', label=f'{label} p99.9')

axes[1].set_xscale('log')
axes[1].set_xlabel('packet loss rate')
axes[1].set_ylabel('latency (ms)')
axes[1].set_title('Where the tail breaks')
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

The right-hand plot has a knee rather than a slope, and the knee is where the affected fraction of
requests crosses the percentile you are measuring. Below it the loss is hidden inside the tail you
already have; above it the percentile jumps by the full timer interval and then stops rising,
because a request rarely loses twice. Which knee you hit first depends on the response size: a
large response reaches it at a tenth of the loss rate a small one needs.

## What the floor on the timer is worth

In [ ]:
def p99_with_rto(min_rto_ms, packet_loss=0.002, packets=70, n=200_000):
    global MIN_RTO_MS
    saved, MIN_RTO_MS = MIN_RTO_MS, min_rto_ms
    try:
        sample, _, _ = simulate_requests(n=n, packets=packets, packet_loss=packet_loss)
        return np.percentile(sample, 99), np.percentile(sample, 99.9)
    finally:
        MIN_RTO_MS = saved

print('0.2% loss, 100 KB response, 40ms round trip')
print(f"{'min RTO':>10}{'p99':>10}{'p99.9':>10}")
for rto in [40, 100, 200, 1000, 3000]:
    p99, p999 = p99_with_rto(rto)
    print(f'{rto:>8} ms{p99:>8.1f}{p999:>10.1f}')

The last row is not hypothetical. A stack that has not yet measured a round trip starts with a
timer of a second or three, which is why the very first loss on a brand-new connection is the most
expensive packet loss a request can suffer. It is also why a connection that is already warm is
worth so much more than its handshake saving alone suggests.

## Variations: a retry, and the loss that cannot be hidden

In [ ]:
def with_hedge(latency, hedge_at_ms):
    """Send a second copy of any request still outstanding at `hedge_at_ms`.

    The hedge is an independent draw, so the result is the faster of the two.
    """
    slow = latency > hedge_at_ms
    second, _, _ = simulate_requests(n=int(slow.sum()), packets=70, packet_loss=0.002)
    hedged = latency.copy()
    hedged[slow] = np.minimum(latency[slow], hedge_at_ms + second)
    return hedged, slow.mean()

base, _, _ = simulate_requests(n=200_000, packets=70, packet_loss=0.002)
print(f'no hedge          p99 {np.percentile(base, 99):>7.1f}  p99.9 {np.percentile(base, 99.9):>7.1f}')
for hedge_at in [np.percentile(base, 95), np.percentile(base, 99), np.percentile(base, 99.9)]:
    hedged, extra = with_hedge(base, hedge_at)
    print(f'hedge at {hedge_at:>6.1f} ms  p99 {np.percentile(hedged, 99):>7.1f}'
          f'  p99.9 {np.percentile(hedged, 99.9):>7.1f}   extra traffic {extra:>6.2%}')

A hedge sent at the 99th percentile costs 1% more traffic and removes most of the timer population
from the tail, because the second copy does not have to wait for the first one's timer. That trade
is usually worth making, and it is strictly better than a blanket retry, which pays for every
request rather than the slow ones.

What a hedge cannot fix is loss on the request path of something that is not idempotent, and what
it must not do is fire during an overload: a hedge at the 99th percentile is 1% extra traffic when
things are healthy and 100% extra when everything is slow, so it needs a rate limit of its own.

## What to try next

- Change `tail_loss_share` directly and watch the percentiles. It is the one parameter here that a
  protocol designer can influence, and it is the reason a transfer that keeps a few extra packets
  in flight at the end recovers faster than a tightly windowed one.
- Model head-of-line blocking: with several logical streams on one connection, a single lost packet
  stalls every stream behind it. Multiply the affected fraction by the number of streams sharing
  the connection and the tail gets worse rather than better, which is the problem a newer transport
  was designed to solve.
- Compare against the measured distribution in the latency-percentiles notebook in this directory,
  and see whether the shoulder in your own data sits at your stack's minimum timer value. If it
  does, you have found the cause rather than a correlation.